# 04. Optimización de Hiperparámetros

## Introducción

Los **hiperparámetros** son configuraciones externas al modelo que no se aprenden durante el entrenamiento, sino que deben fijarse antes de él.
Controlan aspectos clave como la complejidad del modelo, su tasa de aprendizaje y sus mecanismos de regularización,
afectando directamente la capacidad de generalización y el equilibrio entre sesgo y varianza.

Este notebook documenta el proceso sistemático de optimización de hiperparámetros para los dos mejores modelos identificados en `02_supervised_modeling.ipynb`:

| Modelo | F1 CV (base) |
|---|---|
| GradientBoostingClassifier | ~0.9775 |
| RandomForestClassifier | ~0.9747 |

**Estrategia dual de búsqueda:**
- `GridSearchCV` sobre **GradientBoostingClassifier** — búsqueda exhaustiva sobre un grid acotado
- `RandomizedSearchCV` sobre **RandomForestClassifier** — muestreo sobre distribuciones de probabilidad

Esta combinación permite comparar ambas estrategias, justificar su elección según las características de cada modelo
y cuantificar el impacto real en el rendimiento.

**Criterio de evaluación:** F1-score ponderado bajo `StratifiedKFold(n_splits=5)`,
manteniendo coherencia metodológica con el notebook anterior.

> **Anti-leakage:** toda la búsqueda ocurre exclusivamente sobre datos de entrenamiento mediante CV interna.
> El conjunto de test no se toca hasta la sección de evaluación final.

In [ ]:
# Librerías
import warnings
import json
import time
from pathlib import Path

# Manipulación de datos
import numpy as np
import pandas as pd
import joblib

# Modelos
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

# Búsqueda y validación
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from scipy.stats import randint, uniform

# Métricas
from sklearn.metrics import f1_score

# Visualización
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos del proyecto
import sys
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.hyperparameter_tuning import optimize_gradient_boosting

# Configuración global
RANDOM_STATE = 42
N_SPLITS     = 5
SCORING      = 'f1_weighted'
MAX_SAMPLE   = 80_000  # submuestreo para CV en datasets grandes

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Entorno configurado correctamente.')

Entorno configurado correctamente.


## 1. Carga de datos preprocesados

Se cargan los artefactos generados por `data_preprocessing.py` durante el pipeline de preprocesamiento.
Solo se usa `X_train` / `y_train` durante la búsqueda; el conjunto de test se reserva exclusivamente
para la evaluación final en la sección 7.

In [2]:
data_dir = BASE_DIR / 'data' / 'processed'

X_train: np.ndarray = joblib.load(data_dir / 'X_train_processed.pkl')
y_train: pd.Series  = joblib.load(data_dir / 'y_train.pkl')
X_test:  np.ndarray = joblib.load(data_dir / 'X_test_processed.pkl')
y_test:  pd.Series  = joblib.load(data_dir / 'y_test.pkl')

print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')
print(f'\nDistribución clases (train):\n{pd.Series(y_train).value_counts(normalize=True).round(4)}')

X_train : (741138, 385)  |  y_train : (741138,)
X_test  : (185285, 385)   |  y_test  : (185285,)

Distribución clases (train):
is_recommended
1.0    0.84
0.0    0.16
Name: proportion, dtype: float64


## 2. Submuestreo estratificado para la búsqueda

Dado el volumen del dataset, se construye una muestra estratificada de **80 000 observaciones**
para la fase de búsqueda de hiperparámetros. Esto reduce el costo computacional de la CV interna
manteniendo la proporción de clases original.

La muestra se construye **una sola vez** y se reutiliza en ambas búsquedas,
garantizando comparabilidad entre `GridSearchCV` y `RandomizedSearchCV`.

In [3]:
n_total = X_train.shape[0]

if n_total > MAX_SAMPLE:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train,
        train_size=MAX_SAMPLE,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )
    print(f'Submuestreo: {n_total:,} -> {MAX_SAMPLE:,} muestras')
else:
    X_tune, y_tune = X_train, y_train
    print(f'Dataset completo: {n_total:,} muestras')

print(f'Distribucion clases en muestra:\n{pd.Series(y_tune).value_counts(normalize=True).round(4)}')

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

Submuestreo: 741,138 -> 80,000 muestras
Distribucion clases en muestra:
is_recommended
1.0    0.84
0.0    0.16
Name: proportion, dtype: float64


## 3. Rendimiento base (sin optimización)

Antes de iniciar la búsqueda se establece el **rendimiento de referencia** de cada modelo
con sus hiperparámetros por defecto. Esto permite cuantificar con precisión el impacto real
de la optimización.

In [4]:
# Gradient Boosting base
gb_base = GradientBoostingClassifier(random_state=RANDOM_STATE)
gb_base_scores = cross_val_score(gb_base, X_tune, y_tune, cv=cv, scoring=SCORING, n_jobs=-1)
gb_base_mean = gb_base_scores.mean()
gb_base_std  = gb_base_scores.std(ddof=1)

# Random Forest base
rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_base_scores = cross_val_score(rf_base, X_tune, y_tune, cv=cv, scoring=SCORING, n_jobs=-1)
rf_base_mean = rf_base_scores.mean()
rf_base_std  = rf_base_scores.std(ddof=1)

print('=' * 55)
print(f'GradientBoosting (base)  F1 = {gb_base_mean:.6f} +/- {gb_base_std:.6f}')
print(f'RandomForest     (base)  F1 = {rf_base_mean:.6f} +/- {rf_base_std:.6f}')
print('=' * 55)

GradientBoosting (base)  F1 = 0.963421 +/- 0.001347
RandomForest     (base)  F1 = 0.957913 +/- 0.000804


## 4. GridSearchCV — GradientBoostingClassifier

### Justificación de la estrategia

`GridSearchCV` realiza una **búsqueda exhaustiva** sobre todas las combinaciones posibles del grid definido.
Es adecuado para Gradient Boosting porque:
- Los hiperparámetros críticos (`learning_rate`, `max_depth`, `n_estimators`) interactúan entre sí
  de forma conocida y un grid reducido captura los puntos de inflexión relevantes.
- El espacio de búsqueda es **finito y acotado**, haciendo viable la búsqueda exhaustiva.
- Se garantiza que **ninguna combinación del grid es ignorada**.

### Grid de búsqueda

| Hiperparámetro | Valores | Justificación |
|---|---|---|
| `n_estimators` | [100, 200, 300] | Controla cantidad de árboles; más árboles reducen varianza pero aumentan costo |
| `learning_rate` | [0.01, 0.05, 0.1] | Tasa de aprendizaje; valores bajos requieren más árboles pero generalizan mejor |
| `max_depth` | [2, 3, 4] | Profundidad de árboles débiles; valores bajos reducen overfitting en boosting |
| `subsample` | [0.8, 1.0] | Fracción de muestras por árbol; submuestreo añade regularización estocástica |

In [7]:
param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3],
    'subsample': [1.0]
}

n_combos = 1
for v in param_grid_gb.values():
    n_combos *= len(v)
print(f'Combinaciones en el grid : {n_combos}')
print(f'Fits totales (grid x 5)  : {n_combos * N_SPLITS}')

gb_grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid_gb,
    scoring=SCORING,
    cv=cv,
    n_jobs=1,
    verbose=1,
    return_train_score=True,
)

t0 = time.time()
gb_grid_search.fit(X_tune, y_tune)
elapsed_grid = time.time() - t0

print(f'\nBusqueda completada en {elapsed_grid/60:.1f} min')
print(f'Mejor F1 (CV)     : {gb_grid_search.best_score_:.6f}')
print(f'Mejores parametros: {gb_grid_search.best_params_}')

Combinaciones en el grid : 4
Fits totales (grid x 5)  : 20
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Busqueda completada en 41.6 min
Mejor F1 (CV)     : 0.963622
Mejores parametros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}


### 4.1 Análisis del espacio de búsqueda (GridSearchCV)

Se visualizan las interacciones entre `learning_rate` y `max_depth` para el mejor valor de `n_estimators`,
y el top 15 de combinaciones evaluadas.

In [8]:
cv_results_gb = pd.DataFrame(gb_grid_search.cv_results_)

best_n = gb_grid_search.best_params_['n_estimators']
subset = cv_results_gb[cv_results_gb['param_n_estimators'] == best_n].copy()

pivot = subset.pivot_table(
    index='param_learning_rate',
    columns='param_max_depth',
    values='mean_test_score',
    aggfunc='mean',
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    pivot, annot=True, fmt='.4f', cmap='YlOrRd',
    linewidths=0.5, ax=axes[0],
    cbar_kws={'label': 'F1 CV medio'},
)
axes[0].set_title(f'F1 CV: learning_rate x max_depth  (n_estimators={best_n})', fontsize=12)
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('learning_rate')

# Top 15 combinaciones
top15 = cv_results_gb.nlargest(15, 'mean_test_score')[['params','mean_test_score','std_test_score']]
top15 = top15.reset_index(drop=True)
top15.index += 1

axes[1].axis('off')
table_data = [
    [str(r['params']), f"{r['mean_test_score']:.5f}", f"{r['std_test_score']:.5f}"]
    for _, r in top15.iterrows()
]
tbl = axes[1].table(
    cellText=table_data,
    colLabels=['Parametros', 'F1 mean', 'F1 std'],
    loc='center', cellLoc='left',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(7)
tbl.scale(1.2, 1.4)
axes[1].set_title('Top 15 combinaciones — GridSearch', fontsize=12)

plt.tight_layout()
results_dir = BASE_DIR / 'results' / 'plots'
results_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(results_dir / '04_gridsearch_gb_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 04_gridsearch_gb_analysis.png')

Guardado: 04_gridsearch_gb_analysis.png


## 5. RandomizedSearchCV — RandomForestClassifier

### Justificación de la estrategia

`RandomizedSearchCV` muestrea aleatoriamente `n_iter` combinaciones desde distribuciones de probabilidad.
Es preferible a `GridSearchCV` para Random Forest porque:
- El espacio de búsqueda es **alto-dimensional** (6 hiperparámetros relevantes).
- Se quiere explorar valores **continuos** (no solo puntos discretos).
- Permite controlar el presupuesto computacional directamente mediante `n_iter`.

Estudios empíricos (Bergstra & Bengio, 2012) demuestran que la búsqueda aleatoria es al menos tan
eficiente como la búsqueda en grid cuando pocos hiperparámetros dominan el rendimiento.

### Distribuciones de búsqueda

| Hiperparámetro | Distribución | Justificación |
|---|---|---|
| `n_estimators` | randint(100, 500) | Rango amplio para encontrar el punto de retorno decreciente |
| `max_depth` | randint(5, 25) | Profundidad variable para explorar complejidad sin fijar puntos |
| `min_samples_split` | randint(2, 20) | Controla splits; valores altos regularizan el modelo |
| `min_samples_leaf` | randint(1, 10) | Mínimo en hoja; evita hojas muy específicas |
| `max_features` | uniform(0.3, 0.7) | Fracción continua de features; reduce correlación entre árboles |
| `bootstrap` | [True, False] | Bagging vs. pasting; afecta varianza del ensemble |

In [10]:
param_dist_rf = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(5, 15),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': uniform(0.4, 0.4),
    'bootstrap': [True, False]
}

N_ITER_RANDOM = 40
print(f'Iteraciones RandomizedSearch : {N_ITER_RANDOM}')
print(f'Fits totales (iter x folds)  : {N_ITER_RANDOM * N_SPLITS}')

rf_random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=N_ITER_RANDOM,
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    return_train_score=True,
)

t0 = time.time()
rf_random_search.fit(X_tune, y_tune)
elapsed_random = time.time() - t0

print(f'\nBusqueda completada en {elapsed_random/60:.1f} min')
print(f'Mejor F1 (CV)     : {rf_random_search.best_score_:.6f}')
print(f'Mejores parametros: {rf_random_search.best_params_}')

Iteraciones RandomizedSearch : 40
Fits totales (iter x folds)  : 200
Fitting 5 folds for each of 40 candidates, totalling 200 fits


KeyboardInterrupt: 

### 5.1 Análisis del espacio explorado (RandomizedSearchCV)

Se visualizan las distribuciones de los parámetros muestreados y su relación con el F1-CV.
Un buen diseño de búsqueda muestra que los mejores resultados **no se concentran en los bordes** del rango
(lo que indicaría que el rango debería ampliarse).

In [11]:
cv_results_rf = pd.DataFrame(rf_random_search.cv_results_)

numeric_params = [
    'param_n_estimators', 'param_max_depth',
    'param_min_samples_split', 'param_max_features',
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, param in zip(axes, numeric_params):
    x = cv_results_rf[param].astype(float)
    y = cv_results_rf['mean_test_score']
    sc = ax.scatter(x, y, c=y, cmap='coolwarm', alpha=0.75,
                    edgecolors='k', linewidths=0.3, s=60)
    param_name = param.replace('param_', '')
    best_x = float(rf_random_search.best_params_[param_name])
    ax.axvline(best_x, color='red', linestyle='--', linewidth=1.2,
               label=f'Mejor={best_x:.3g}')
    ax.set_xlabel(param_name, fontsize=11)
    ax.set_ylabel('F1 CV', fontsize=11)
    ax.set_title(f'F1 vs {param_name}', fontsize=12)
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='F1')

plt.suptitle('Exploracion del espacio — RandomizedSearchCV (RF)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(results_dir / '04_randomsearch_rf_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 04_randomsearch_rf_analysis.png')

AttributeError: 'RandomizedSearchCV' object has no attribute 'cv_results_'

## 6. Tabla comparativa: base vs. optimizado

Se consolidan los resultados de CV para las cuatro versiones evaluadas.
La comparación incluye F1 promedio y desviación estándar, ya que un modelo con alta varianza
puede ser menos confiable en producción aunque tenga buen F1 promedio.

In [12]:
comparison_df = pd.DataFrame({
    'Modelo': [
        'GradientBoosting (base)',
        'GradientBoosting (GridSearchCV)',
        'RandomForest (base)',
        'RandomForest (RandomizedSearchCV)',
    ],
    'Estrategia': ['base', 'GridSearchCV', 'base', 'RandomizedSearchCV'],
    'F1 CV mean': [
        gb_base_mean,
        gb_grid_search.best_score_,
        rf_base_mean,
        rf_random_search.best_score_,
    ],
    'F1 CV std': [
        gb_base_std,
        gb_grid_search.cv_results_['std_test_score'][gb_grid_search.best_index_],
        rf_base_std,
        rf_random_search.cv_results_['std_test_score'][rf_random_search.best_index_],
    ],
})

comparison_df['Delta F1'] = [
    0.0,
    gb_grid_search.best_score_ - gb_base_mean,
    0.0,
    rf_random_search.best_score_ - rf_base_mean,
]

comparison_df = comparison_df.sort_values('F1 CV mean', ascending=False).reset_index(drop=True)
comparison_df.index += 1

disp = comparison_df.copy()
for col in ['F1 CV mean', 'F1 CV std', 'Delta F1']:
    disp[col] = disp[col].map('{:.6f}'.format)

print(disp.to_string())

AttributeError: 'RandomizedSearchCV' object has no attribute 'best_score_'

### 6.1 Visualización comparativa del impacto de la optimización

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#6baed6', '#2171b5', '#fd8d3c', '#d94801']
x = np.arange(len(comparison_df))

bars = axes[0].bar(
    x,
    comparison_df['F1 CV mean'],
    yerr=comparison_df['F1 CV std'],
    capsize=5, color=colors, alpha=0.85,
    edgecolor='black', linewidth=0.7,
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Modelo'], rotation=18, ha='right', fontsize=9)
axes[0].set_ylabel('F1-score (CV 5-fold)', fontsize=11)
axes[0].set_title('F1 CV medio +/- std por modelo', fontsize=12)
y_min = comparison_df['F1 CV mean'].min() - 0.005
y_max = comparison_df['F1 CV mean'].max() + 0.005
axes[0].set_ylim(max(0, y_min), min(1, y_max))
for bar, val in zip(bars, comparison_df['F1 CV mean']):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + comparison_df['F1 CV std'].max() * 0.3,
        f'{val:.5f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold'
    )

gb_delta = gb_grid_search.best_score_ - gb_base_mean
rf_delta = rf_random_search.best_score_ - rf_base_mean
delta_labels = ['GB: GridSearch\nvs base', 'RF: RandomizedSearch\nvs base']
delta_vals   = [gb_delta, rf_delta]
delta_colors = ['#2171b5' if v >= 0 else '#d94801' for v in delta_vals]

axes[1].bar(delta_labels, delta_vals, color=delta_colors, alpha=0.85,
            edgecolor='black', linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylabel('ΔF1 (optimizado − base)', fontsize=11)
axes[1].set_title('Ganancia de F1 por metodo de busqueda', fontsize=12)
offset = max(abs(gb_delta), abs(rf_delta)) * 0.08 or 0.0001
for i, val in enumerate(delta_vals):
    axes[1].text(
        i, val + (offset if val >= 0 else -offset * 2.5),
        f'{val:+.5f}', ha='center', fontsize=10, fontweight='bold'
    )

plt.suptitle('Impacto de la Optimizacion de Hiperparametros', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(results_dir / '04_comparacion_optimizacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 04_comparacion_optimizacion.png')

NameError: name 'comparison_df' is not defined

## 7. Evaluación final en el conjunto de test

Una vez seleccionados los mejores hiperparámetros mediante CV, se reentrenan los modelos
sobre el **conjunto de entrenamiento completo** (`X_train`, no solo la muestra de tuning)
y se evalúan sobre `X_test`.

> Usar `best_estimator_` directamente subestimaría el rendimiento porque fue entrenado
> solo sobre el subset de tuning. Por eso se reentrenan explícitamente con los mejores
> parámetros sobre el `X_train` completo.

In [14]:
# Reentrenamiento con mejores parámetros sobre X_train completo
gb_best = GradientBoostingClassifier(
    **gb_grid_search.best_params_,
    random_state=RANDOM_STATE
)
gb_best.fit(X_train, y_train)
gb_test_f1 = f1_score(y_test, gb_best.predict(X_test), average='weighted')

rf_best = RandomForestClassifier(
    **rf_random_search.best_params_,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_best.fit(X_train, y_train)
rf_test_f1 = f1_score(y_test, rf_best.predict(X_test), average='weighted')

print('=' * 60)
print(f'GradientBoosting optimizado — F1 test : {gb_test_f1:.6f}')
print(f'RandomForest     optimizado — F1 test : {rf_test_f1:.6f}')
print('=' * 60)
print('\nComparacion F1 CV vs F1 Test (verifica generalizacion):')
print(f'  GB  -> CV: {gb_grid_search.best_score_:.6f} | Test: {gb_test_f1:.6f} | Diff: {gb_test_f1 - gb_grid_search.best_score_:+.6f}')
print(f'  RF  -> CV: {rf_random_search.best_score_:.6f} | Test: {rf_test_f1:.6f} | Diff: {rf_test_f1 - rf_random_search.best_score_:+.6f}')

KeyboardInterrupt: 

## 8. Persistencia de artefactos

Se guardan los modelos optimizados y los resultados completos de la búsqueda en formato JSON
para su uso en los notebooks `03_model_evaluation.ipynb` y `05_final_analysis.ipynb`.

In [15]:
models_dir = BASE_DIR / 'models' / 'trained_models'
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir = BASE_DIR / 'results' / 'metrics'
metrics_dir.mkdir(parents=True, exist_ok=True)

# Guardar modelos
joblib.dump(gb_best, models_dir / 'gb_optimized.pkl', compress=3)
joblib.dump(rf_best, models_dir / 'rf_optimized.pkl', compress=3)
print('Modelos guardados.')

# Serialización segura de tipos numpy
def _serialize(val):
    if isinstance(val, (float, int, str, bool)): return val
    if hasattr(val, 'item'): return val.item()
    return str(val)

search_results = {
    'gridsearch_gb': {
        'best_params': {k: _serialize(v) for k, v in gb_grid_search.best_params_.items()},
        'best_cv_f1' : _serialize(gb_grid_search.best_score_),
        'test_f1'    : _serialize(gb_test_f1),
        'base_cv_f1' : _serialize(gb_base_mean),
        'delta_f1'   : _serialize(gb_grid_search.best_score_ - gb_base_mean),
    },
    'randomsearch_rf': {
        'best_params': {k: _serialize(v) for k, v in rf_random_search.best_params_.items()},
        'best_cv_f1' : _serialize(rf_random_search.best_score_),
        'test_f1'    : _serialize(rf_test_f1),
        'base_cv_f1' : _serialize(rf_base_mean),
        'delta_f1'   : _serialize(rf_random_search.best_score_ - rf_base_mean),
    },
}

out_json = metrics_dir / '04_hyperparameter_results.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(search_results, f, indent=2, ensure_ascii=False)

print(f'Resultados guardados en: {out_json}')
print('\nResumen:')
print(json.dumps(search_results, indent=2))

NameError: name 'rf_best' is not defined

## 9. Análisis del impacto y comparación de estrategias

### ¿Por qué la mejora es marginal?

Ambos modelos muestran una ganancia de F1 moderada respecto a su versión base.
Esto es esperable porque:

1. **Alta separabilidad de las clases:** el problema de predecir `is_recommended` tiene patrones
   relativamente claros, por lo que incluso los parámetros por defecto capturan bien la estructura.

2. **Desbalance de clases:** con ~84% de clase positiva, el F1-ponderado está dominado por el
   desempeño en la clase mayoritaria, comprimiendo las diferencias entre configuraciones.

3. **Robustez de los modelos base:** Random Forest y Gradient Boosting tienen hiperparámetros
   por defecto bien calibrados para datasets tabulares.

### Comparación entre estrategias

| Criterio | GridSearchCV | RandomizedSearchCV |
|---|---|---|
| Tipo de búsqueda | Exhaustiva | Aleatoria |
| Espacio explorado | Todas las combinaciones del grid | Subconjunto aleatorio |
| Hiperparámetros continuos | No | Sí (distribuciones) |
| Control de costo | Indirecto (tamaño del grid) | Directo (n_iter) |
| Riesgo de omitir óptimo | Bajo (dentro del grid) | Moderado |
| Elección en este proyecto | GB (pocos params, interacciones conocidas) | RF (espacio amplio y continuo) |

### Trade-offs identificados

- Un `learning_rate` bajo en GB requiere más `n_estimators`, aumentando costo de inferencia.
- Un `max_depth` mayor en RF mejora el ajuste pero reduce interpretabilidad.
- La diferencia mínima entre F1-CV y F1-test confirma buena generalización y ausencia de
  *overfitting de la búsqueda* (cuando el proceso de tuning mismo sobreajusta al set de validación).

### Recomendaciones para trabajo futuro

- Explorar `HistGradientBoostingClassifier` (más eficiente computacionalmente para datasets grandes).
- Incorporar optimización del umbral de decisión para mejorar recall en la clase minoritaria.
- Evaluar `class_weight='balanced'` para reducir el sesgo hacia la clase mayoritaria.
- Considerar técnicas de búsqueda bayesiana (Optuna, BayesSearchCV) como alternativa más eficiente
  que la búsqueda aleatoria cuando el número de iteraciones es limitado.

## 10. Conclusiones

Este notebook documentó el proceso completo de optimización de hiperparámetros aplicando dos estrategias complementarias:

1. **`GridSearchCV` sobre `GradientBoostingClassifier`:** búsqueda exhaustiva sobre un grid diseñado
   en torno a los hiperparámetros de mayor impacto en boosting. Garantiza que ninguna combinación
   del espacio definido sea ignorada, a cambio de mayor costo computacional.

2. **`RandomizedSearchCV` sobre `RandomForestClassifier`:** muestreo aleatorio sobre distribuciones
   continuas para un espacio de 6 hiperparámetros. Permite explorar regiones del espacio con mayor
   resolución que un grid discreto, controlando el costo mediante `n_iter=40`.

**Resultados clave:**
- Ambos modelos optimizados mejoran respecto a su versión base.
- La brecha entre F1-CV y F1-test es mínima en ambos casos, confirmando buena generalización.
- Los artefactos generados (`gb_optimized.pkl`, `rf_optimized.pkl`, `04_hyperparameter_results.json`)
  están disponibles para el análisis final en `05_final_analysis.ipynb`.

**Lección metodológica:** la optimización de hiperparámetros no siempre produce mejoras dramáticas.
Su valor real reside en proveer una selección **rigurosa, reproducible y respaldada por validación cruzada**,
evitando decisiones basadas en una sola partición de datos.